# ==============================================================================
# 🩺 METABOLIC HEALTHCARE: DIABETES RISK PREDICTION VIA SUPPORT VECTOR CLASSIFIER
# ==============================================================================
### Core Principles:
Predicting diabetes risk from demographic factors (`gender`, `age`), comorbidities (`hypertension`, `heart_disease`), and blood biomarkers (`bmi`, `HbA1c_level`, `blood_glucose_level`).

Combines `ColumnTransformer` with `OneHotEncoder` and `StandardScaler` to ensure support vector margins are not distorted by numerical biomarker ranges.


In [1]:
# STEP 1: ENVIRONMENT SETUP
import os
import time
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, accuracy_score, recall_score

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("✅ STEP 1: Metabolic Healthcare Environment ready.")


✅ STEP 1: Metabolic Healthcare Environment ready.


## 📥 STEP 2 & 3: INGESTION & DATASET HARMONIZATION
Loading Kaggle Diabetes Prediction Dataset (100,000 patient records).
Sampling 15,000 patients for responsive non-linear SVM cross-validation.


In [2]:
# STEP 2 & 3: LOAD & SAMPLE
data_path = 'data/diabetes/diabetes_prediction_dataset.csv'
df = pd.read_csv(data_path)

# Sample 15,000 patients stratified by diabetes for responsive quadratic SVM computation
df_sample = df.sample(n=15000, random_state=42).reset_index(drop=True)

df_sample.columns = [c.strip().lower() for c in df_sample.columns]
print(f"📊 Dataset Sample Shape: {df_sample.shape[0]:,} patients, {df_sample.shape[1]} attributes")
print(f"Diabetes Prevalence:\n{df_sample['diabetes'].value_counts(normalize=True).round(4)*100}%")
df_sample.head(3)


📊 Dataset Sample Shape: 15,000 patients, 9 attributes
Diabetes Prevalence:
diabetes
0    91.44
1     8.56
Name: proportion, dtype: float64%


,gender,age,hypertension,heart_disease,smoking_history,bmi,hba1c_level,blood_glucose_level,diabetes
0,Female,13.0,0,0,No Info,20.82,5.8,126,0
1,Female,3.0,0,0,No Info,21.00,5.0,145,0
2,Male,63.0,0,0,former,25.32,3.5,200,0


## 🧹 STEP 4 & 5: SEGREGATION & HYGIENE
Separating features and target `diabetes`.


In [3]:
# STEP 4 & 5: SEGREGATION
X = df_sample.drop(columns=['diabetes'])
y = df_sample['diabetes']

print(f"Missing attributes: {X.isnull().sum().sum()}")


Missing attributes: 0


## 🔒 STEP 6: STRATIFIED TRAIN-TEST SPLIT
80/20 stratified split.


In [4]:
# STEP 6: STRATIFIED SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train Patients: {X_train.shape[0]:,} ({y_train.sum()} Diabetics)")
print(f"Test Patients : {X_test.shape[0]:,} ({y_test.sum()} Diabetics)")


Train Patients: 12,000 (1027 Diabetics)
Test Patients : 3,000 (257 Diabetics)


## ⚙️ STEP 7: HETEROGENEOUS COLUMN TRANSFORMER PREPROCESSOR
Encoding categorical smoking history and gender alongside standardized physiological indicators.


In [5]:
# STEP 7: PREPROCESSOR
cat_cols = ['gender', 'smoking_history']
num_cols = ['age', 'hypertension', 'heart_disease', 'bmi', 'hba1c_level', 'blood_glucose_level']

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False), cat_cols),
    ('num', StandardScaler(), num_cols)
])
print("✅ STEP 7: ColumnTransformer Preprocessor configured.")


✅ STEP 7: ColumnTransformer Preprocessor configured.


## ⚠️ STEP 8: BASELINE FIT (Linear vs RBF SVC)
Benchmarking linear vs non-linear metabolic risk boundaries.


In [6]:
# STEP 8: BASELINE DUEL
from sklearn.svm import LinearSVC

pipe_lin = Pipeline([
    ('prep', preprocessor),
    ('svc', LinearSVC(dual=False, random_state=42))
])
pipe_lin.fit(X_train, y_train)

pipe_rbf = Pipeline([
    ('prep', preprocessor),
    ('svc', SVC(kernel='rbf', class_weight='balanced', random_state=42))
])
pipe_rbf.fit(X_train, y_train)

rec_lin = recall_score(y_test, pipe_lin.predict(X_test))
rec_rbf = recall_score(y_test, pipe_rbf.predict(X_test))

print("="*65)
print(f"LinearSVC Recall          : {rec_lin*100:.2f}%")
print(f"RBF SVC (Balanced) Recall : {rec_rbf*100:.2f}%")
print(f"Non-Linear Sensitivity Gain: +{(rec_rbf - rec_lin)*100:.2f}% Patients detected!")
print("="*65)


LinearSVC Recall          : 56.81%
RBF SVC (Balanced) Recall : 86.77%
Non-Linear Sensitivity Gain: +29.96% Patients detected!


## 🎯 STEP 9: TUNING HYPERPARAMETERS VIA 3-FOLD CV
Optimizing $C$ and $\gamma$ for clinical recall.


In [7]:
# STEP 9: GRID SEARCH
param_grid = {
    'svc__C': [1, 10],
    'svc__gamma': ['scale', 0.05]
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
pipe_tune = Pipeline([
    ('prep', preprocessor),
    ('svc', SVC(kernel='rbf', class_weight='balanced', random_state=42))
])

grid = GridSearchCV(pipe_tune, param_grid=param_grid, cv=cv, scoring='recall', n_jobs=-1)
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
print(f"🏆 Best Clinical Hyperparameters: {grid.best_params_}")
print(f"🎯 Best 3-Fold CV Recall: {grid.best_score_*100:.2f}%")


🏆 Best Clinical Hyperparameters: {'svc__C': 1, 'svc__gamma': 0.05}
🎯 Best 3-Fold CV Recall: 88.51%


## 📊 STEP 10 & 11: METABOLIC DIAGNOSTIC REPORT & CONFUSION MATRIX
Reviewing clinical recall and screening metrics.


In [8]:
# STEP 10 & 11: EVALUATION REPORT
y_pred = best_model.predict(X_test)
cm = confusion_matrix(y_test, y_pred)

print("📋 METABOLIC SCREENING REPORT:")
print(classification_report(y_test, y_pred, target_names=['Non-Diabetic', 'Diabetic']))

print(f"🎯 Test Accuracy   : {accuracy_score(y_test, y_pred)*100:.2f}%")
print(f"🎯 Diabetic Recall : {recall_score(y_test, y_pred)*100:.2f}%")

print(f"\nConfusion Matrix:\n{cm}")
tn, fp, fn, tp = cm.ravel()
print(f"Detected Diabetics : {tp}/{tp+fn}")
print(f"Missed Diabetics   : {fn}")


📋 METABOLIC SCREENING REPORT:
              precision    recall  f1-score   support

Non-Diabetic       0.99      0.89      0.93      2743
    Diabetic       0.42      0.88      0.57       257

    accuracy                           0.89      3000
   macro avg       0.71      0.89      0.75      3000
weighted avg       0.94      0.89      0.90      3000

🎯 Test Accuracy   : 88.70%
🎯 Diabetic Recall : 88.33%

Confusion Matrix:
[[2434  309]
 [  30  227]]
Detected Diabetics : 227/257
Missed Diabetics   : 30


## 💾 STEP 12 & 13: PRODUCTION SERIALIZATION & CLINICAL TRIAGE SMOKE TEST
Deploying the pipeline and testing electronic health record (EHR) screening latency.


In [9]:
# STEP 12 & 13: SERIALIZATION & SMOKE TEST
os.makedirs('production_models', exist_ok=True)
model_path = 'production_models/diabetes_risk_svc_pipeline.joblib'
joblib.dump(best_model, model_path, compress=3)
print(f"💾 Production pipeline saved to: {model_path} ({os.path.getsize(model_path):,} bytes)")

loaded_pipe = joblib.load(model_path)
sample_patient = X_test.iloc[[0]]

t0 = time.perf_counter()
pred_risk = loaded_pipe.predict(sample_patient)[0]
latency_ms = (time.perf_counter() - t0) * 1000

print(f"\n🩺 LIVE EHR CLINICAL TRIAGE:")
print(f"   Diagnostic Screening : {'🚨 HIGH DIABETES RISK' if pred_risk == 1 else '🟢 LOW / NORMAL RISK'}")
print(f"   Screening Latency    : {latency_ms:.3f} ms (SLA < 2.0ms: {'PASS' if latency_ms < 2.0 else 'CHECK'})")


💾 Production pipeline saved to: production_models/diabetes_risk_svc_pipeline.joblib (56,297 bytes)

🩺 LIVE EHR CLINICAL TRIAGE:
   Diagnostic Screening : 🟢 LOW / NORMAL RISK
   Screening Latency    : 4.219 ms (SLA < 2.0ms: CHECK)
